# Детекция ботов по событиям на Авито

Тестовое задание. Есть журнал событий пользователей — 330 тысяч строк: просмотры, поиск, контакты, с временем, платформой, user-agent и координатами курсора. Для каждой куки в заданном временном окне нужно предсказать вероятность того, что это бот.

Метрика — precision при recall 0.7 (P@R70): если ловим 70% ботов, какая доля помеченных действительно боты.

**Итог:** LightGBM на 73 агрегатах по куке, P@R70 = 0.75 ± 0.01 на валидации по времени. Baseline на объёме активности даёт 0.10.

Данные и `metric.py` от организаторов в репозиторий не входят: данные кладутся в `data/`, `metric.py` — рядом с ноутбуком.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from metric import precision_at_recall

PATH = 'data/'

date_cols = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
train = pd.read_csv(PATH + 'train.csv', parse_dates=date_cols)
test = pd.read_csv(PATH + 'test.csv', parse_dates=date_cols)
events = pd.read_csv(PATH + 'events.csv', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', round(train.target.mean(), 4))
print('окна train:', train.window_start_ts.min().date(), '-', train.window_start_ts.max().date())
print('окна test: ', test.window_start_ts.min().date(), '-', test.window_start_ts.max().date())

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811
окна train: 2026-04-06 - 2026-04-19
окна test:  2026-04-20 - 2026-04-26


## Чистка

В файле событий около 1.5% полных дубликатов и разный регистр в `platform`.
Число дубликатов сохраняем — оно ещё пригодится как признак.

In [2]:
print('доля дубликатов:', round(events.duplicated().mean(), 4))
dup_count = events[events.duplicated()].groupby('cookie_id').size().rename('dup_count')
events = events.drop_duplicates()

events['platform'] = events['platform'].str.lower().str.replace('iphone', 'ios', regex=False)
print(events.platform.value_counts())
print()
print('пропуски по колонкам:')
print(events.isna().mean().round(3))

доля дубликатов: 0.0148


platform
web        136786
android    124988
desktop     46157
ios         16111
Name: count, dtype: int64

пропуски по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.349
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: float64


## Окно наблюдения

Признаки считаем по событиям внутри окна: `window_start_ts <= event_ts < window_end_ts`.

In [3]:
def events_in_window(events, meta):
    cols = ['cookie_id', 'cookie_created_at', 'window_start_ts', 'window_end_ts']
    ev = events.merge(meta[cols], on='cookie_id')
    ev = ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]
    return ev.sort_values(['cookie_id', 'event_ts']).reset_index(drop=True)


ev_train = events_in_window(events, train)
ev_test = events_in_window(events, test)
print(len(ev_train), len(ev_test))

195484 88349


## Популярность объявлений

Сколько разных кук открывали объявление. Считаем по дням и берём накопленную сумму строго
до дня окна: более поздние дни на момент предсказания ещё не наступили.

Тот же счётчик по всем дням сразу даёт на валидации около 0.85, но использует будущее
относительно окна, поэтому не используется.

In [4]:
def item_popularity_before(ev_parts):
    ev = pd.concat(ev_parts)[['item_id', 'cookie_id', 'event_ts']].dropna(subset=['item_id'])
    ev['date'] = ev.event_ts.dt.normalize()
    daily = ev.groupby(['item_id', 'date']).cookie_id.nunique().rename('n_cookies').reset_index()
    daily = daily.sort_values(['item_id', 'date'])
    daily['pop_before'] = daily.groupby('item_id').n_cookies.cumsum() - daily.n_cookies
    return daily.set_index(['item_id', 'date']).pop_before


POP_BEFORE = item_popularity_before([ev_train, ev_test])
print(POP_BEFORE.head())

item_id    date      
1000000.0  2026-04-10    0
           2026-04-16    1
1000001.0  2026-04-11    0
1000002.0  2026-04-15    0
           2026-04-18    1
Name: pop_before, dtype: int64


## Признаки

Агрегаты на уровне куки, пять групп:

- тайминги: квантили интервалов между событиями, разброс, доля быстрых переходов, доля
  «круглых» интервалов, доля самого частого интервала;
- время внутри окна: начало, конец и длительность активности, возраст куки;
- содержание: уникальные объявления, категории, локации, повторы, глубина страниц выдачи,
  повторы поисковых запросов;
- окружение: платформа, координаты курсора, флаги по user-agent (headless, HTTP-библиотека
  вместо браузера, расхождение user-agent и `platform`);
- внешние счётчики: популярность объявлений и число дублирующихся строк.

Сырую строку user-agent не берём: это сотни редких значений.

In [5]:
PASSIVE = ['item_view', 'search_results_view', 'seller_page_view']
ENGAGEMENT = ['photo_swipe', 'favorite_add', 'login',
              'contact_phone_show', 'contact_chat_open', 'contact_message_sent']
PLATFORMS = ['web', 'desktop', 'ios', 'android']


def entropy(values):
    p = values.value_counts(normalize=True).values
    return float(-(p * np.log(p + 1e-12)).sum())


def top_share(values):
    if values.notna().any():
        return values.value_counts(normalize=True).max()
    return np.nan

In [6]:
def build_features(ev, meta, pop_before, dup_count):
    ev = ev.copy()
    ev['is_passive'] = ev.event_name.isin(PASSIVE).astype(int)
    ev['is_engagement'] = ev.event_name.isin(ENGAGEMENT).astype(int)
    ev['has_pointer'] = ev.pointer_x.notna() & ev.pointer_y.notna()
    ev['dt'] = ev.groupby('cookie_id').event_ts.diff().dt.total_seconds()

    dx = ev.groupby('cookie_id').pointer_x.diff()
    dy = ev.groupby('cookie_id').pointer_y.diff()
    ev['pointer_dist'] = np.sqrt(dx ** 2 + dy ** 2)

    g = ev.groupby('cookie_id')
    dt = g['dt']
    f = pd.DataFrame(index=g.size().index)

    # объём и разнообразие
    f['event_count'] = g.size()
    f['n_unique_events'] = g.event_name.nunique()
    f['unique_per_action'] = f.n_unique_events / f.event_count
    f['item_nunique'] = g.item_id.nunique()
    f['category_nunique'] = g.item_category.nunique()
    f['location_nunique'] = g.item_location.nunique()
    f['seller_type_nunique'] = g.seller_type.nunique()

    # тип активности
    f['n_passive'] = g.is_passive.sum()
    f['n_engagement'] = g.is_engagement.sum()
    f['passive_ratio'] = f.n_passive / f.event_count
    f['engagement_ratio'] = f.n_engagement / f.event_count
    f['view_to_action'] = f.n_passive / (f.n_engagement + 1)
    f['has_engagement'] = (f.n_engagement > 0).astype(int)

    event_counts = ev.pivot_table(index='cookie_id', columns='event_name',
                                  values='eid', aggfunc='size', fill_value=0)
    for name in event_counts.columns:
        f['share_' + name] = event_counts[name] / f.event_count

    # тайминги
    f['dt_mean'] = dt.mean()
    f['dt_std'] = dt.std()
    f['dt_median'] = dt.median()
    f['dt_min'] = dt.min()
    f['dt_q10'] = dt.quantile(0.10)
    f['dt_q25'] = dt.quantile(0.25)
    f['dt_q90'] = dt.quantile(0.90)
    f['dt_iqr'] = dt.quantile(0.75) - f.dt_q25
    f['dt_cv'] = f.dt_std / f.dt_mean
    f['dt_med_mean_ratio'] = f.dt_median / (f.dt_mean + 1e-9)
    f['burstiness'] = (f.dt_std - f.dt_mean) / (f.dt_std + f.dt_mean)
    f['dt_log_std'] = dt.apply(lambda s: np.log1p(s.dropna()).std())
    f['dt_fast_ratio'] = dt.apply(lambda s: (s < 30).mean())
    f['dt_very_fast_ratio'] = dt.apply(lambda s: (s < 5).mean())
    f['dt_round_ratio'] = dt.apply(lambda s: (s.dropna().round() % 5 == 0).mean())
    f['dt_mode_share'] = dt.apply(lambda s: top_share(s.dropna().round()))

    # время внутри окна
    first_event = g.event_ts.min()
    last_event = g.event_ts.max()
    window_start = g.window_start_ts.first()
    window_end = g.window_end_ts.first()
    cookie_birth = g.cookie_created_at.first()

    f['event_span'] = (last_event - first_event).dt.total_seconds()
    f['event_rate'] = f.event_count / (f.event_span + 1)
    f['start_offset'] = (first_event - window_start).dt.total_seconds()
    f['end_offset'] = (window_end - last_event).dt.total_seconds()
    f['cookie_age_at_window_start'] = (window_start - cookie_birth).dt.total_seconds()

    # содержание
    f['events_per_item'] = f.event_count / (f.item_nunique + 1)
    f['item_repeat_ratio'] = 1 - f.item_nunique / f.event_count
    f['category_entropy'] = g.item_category.apply(entropy)
    f['category_top_share'] = g.item_category.apply(top_share)
    f['search_query_count'] = g.search_query.apply(lambda s: s.notna().sum())
    f['search_query_nunique'] = g.search_query.nunique()
    f['has_search_query'] = (f.search_query_count > 0).astype(int)
    f['query_repeat_ratio'] = 1 - f.search_query_nunique / (f.search_query_count + 1)
    f['search_page_max'] = g.search_page.max()
    f['search_page_mean'] = g.search_page.mean()
    f['deep_page_ratio'] = g.search_page.apply(
        lambda s: (s.dropna() > 1).mean() if s.notna().any() else np.nan)

    # окружение
    platform_share = ev.pivot_table(index='cookie_id', columns='platform',
                                    values='eid', aggfunc='size', fill_value=0)
    platform_share = platform_share.div(platform_share.sum(axis=1), axis=0)
    for p in PLATFORMS:
        if p in platform_share.columns:
            f['platform_' + p + '_share'] = platform_share[p]
    f['platform_nunique'] = g.platform.nunique()
    f['ua_nunique'] = g.user_agent.nunique()
    f['has_pointer_ratio'] = g.has_pointer.mean()
    f['pointer_unique_count'] = g.pointer_x.nunique()
    f['pointer_dist_mean'] = g.pointer_dist.mean()
    f['pointer_dist_std'] = g.pointer_dist.std()
    f['pointer_x_std'] = g.pointer_x.std()
    f['pointer_y_std'] = g.pointer_y.std()

    # user-agent
    ev['ua_headless'] = ev.user_agent.str.contains('HeadlessChrome|PhantomJS|Selenium', case=False)
    ev['ua_library'] = ~ev.user_agent.str.startswith('Mozilla')
    ev['ua_says_mobile'] = ev.user_agent.str.contains('iPhone|Android|Mobile', case=False)
    ev['ua_mismatch'] = ev.ua_says_mobile != ev.platform.isin(['ios', 'android'])
    f['ua_headless_share'] = g.ua_headless.mean()
    f['ua_library_share'] = g.ua_library.mean()
    f['ua_mismatch_share'] = g.ua_mismatch.mean()

    # дубликаты строк
    f['dup_count'] = dup_count.reindex(f.index).fillna(0)
    f['dup_ratio'] = f.dup_count / f.event_count

    # популярность объявлений
    ev['date'] = ev.event_ts.dt.normalize()
    ev['pop_before'] = ev.set_index(['item_id', 'date']).index.map(pop_before)
    f['pop_before_mean'] = g.pop_before.mean()
    f['pop_before_max'] = g.pop_before.max()

    f = f.reset_index()
    f['window_start_ts'] = g.window_start_ts.first().values
    return meta[['cookie_id']].merge(f, on='cookie_id', how='left')

Пропуски: счётчики — нулями, остальное — значением -1. Медиана здесь не подходит, она
считалась бы по train и test отдельно.

In [7]:
def fill_missing(df):
    df = df.copy()
    count_cols = [c for c in df.columns if c.startswith(('n_', 'share_', 'has_'))]
    count_cols += ['event_count', 'search_query_count']
    for col in count_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(-1)
    return df


X_train = fill_missing(build_features(ev_train, train, POP_BEFORE, dup_count))
X_train = X_train.merge(train[['cookie_id', 'target']], on='cookie_id')
X_test = fill_missing(build_features(ev_test, test, POP_BEFORE, dup_count))

FEATURES = [c for c in X_train.columns if c not in ['cookie_id', 'window_start_ts', 'target']]
print(X_train.shape, X_test.shape)
print('признаков:', len(FEATURES))

(11091, 76) (4909, 75)
признаков: 73


## Валидация

Тестовые окна идут после обучающих, поэтому валидация сдвинута во времени: обучаемся на
всём, что раньше даты разреза, проверяемся на трёх следующих днях.

В трёх днях около 160 ботов, разброс между сидами сравним с разницей между моделями,
поэтому берём три даты разреза и три сида в каждой.

In [8]:
SPLIT_DATES = [pd.Timestamp('2026-04-11'), pd.Timestamp('2026-04-14'), pd.Timestamp('2026-04-17')]


def make_model(seed):
    return lgb.LGBMClassifier(
        n_estimators=350,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        objective='binary',
        random_state=seed,
        verbose=-1,
    )


def validate(features, n_seeds=3):
    fold_scores = []
    for split_date in SPLIT_DATES:
        part_train = X_train[X_train.window_start_ts < split_date]
        part_valid = X_train[(X_train.window_start_ts >= split_date)
                             & (X_train.window_start_ts < split_date + pd.Timedelta('3D'))]
        seed_scores = []
        for seed in range(n_seeds):
            model = make_model(seed)
            model.fit(part_train[features], part_train.target)
            proba = model.predict_proba(part_valid[features])[:, 1]
            seed_scores.append(precision_at_recall(part_valid.target, proba, 0.7))
        fold_scores.append(np.mean(seed_scores))
    return np.mean(fold_scores), np.std(fold_scores), fold_scores

In [9]:
baseline_features = ['event_count', 'item_nunique']
timing_features = [c for c in FEATURES if c.startswith('dt_')] + ['event_count', 'event_rate']

for name, features in [('только объём (baseline)', baseline_features),
                       ('объём + тайминги', timing_features),
                       ('все признаки', FEATURES)]:
    mean, std, folds = validate(features)
    print(f'{name:<24} P@R70 = {mean:.4f} +- {std:.4f}   по фолдам: {[round(x, 3) for x in folds]}')

только объём (baseline)  P@R70 = 0.1015 +- 0.0078   по фолдам: [np.float64(0.091), np.float64(0.104), np.float64(0.109)]


объём + тайминги         P@R70 = 0.2926 +- 0.0317   по фолдам: [np.float64(0.279), np.float64(0.262), np.float64(0.336)]


все признаки             P@R70 = 0.7494 +- 0.0088   по фолдам: [np.float64(0.742), np.float64(0.762), np.float64(0.745)]


Ни одна группа признаков не решает задачу в одиночку: тайминги вместе с объёмом дают 0.29, все признаки — 0.75.

## Вклад признаков

In [10]:
model = make_model(0)
model.fit(X_train[FEATURES], X_train.target)

importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(importance.head(15))

dt_median                     386
pop_before_mean               372
cookie_age_at_window_start    331
item_repeat_ratio             308
location_nunique              293
dt_iqr                        291
start_offset                  276
dt_med_mean_ratio             273
category_entropy              270
search_page_mean              265
end_offset                    261
dt_fast_ratio                 256
dt_q90                        248
dt_log_std                    240
dt_q25                        230
dtype: int32


In [11]:
GROUPS = {
    'тайминги': [c for c in FEATURES if c.startswith('dt_')],
    'положение в окне': ['event_span', 'event_rate', 'start_offset', 'end_offset',
                         'cookie_age_at_window_start'],
    'содержание': ['events_per_item', 'item_repeat_ratio', 'category_entropy', 'category_top_share',
                   'search_page_mean', 'deep_page_ratio', 'query_repeat_ratio', 'seller_type_nunique'],
    'окружение': ['ua_nunique', 'platform_nunique', 'pointer_dist_std', 'pointer_x_std',
                  'pointer_y_std', 'has_pointer_ratio', 'ua_headless_share',
                  'ua_library_share', 'ua_mismatch_share'],
    'популярность и дубликаты': ['pop_before_mean', 'pop_before_max', 'dup_count', 'dup_ratio'],
}

full_score = validate(FEATURES)[0]
print(f'все признаки: {full_score:.4f}')
for name, group in GROUPS.items():
    rest = [c for c in FEATURES if c not in group]
    mean, std, _ = validate(rest)
    print(f'без группы "{name}": {mean:.4f} ({mean - full_score:+.4f})')

все признаки: 0.7494


без группы "тайминги": 0.6887 (-0.0606)


без группы "положение в окне": 0.7597 (+0.0103)


без группы "содержание": 0.7609 (+0.0115)


без группы "окружение": 0.7192 (-0.0302)


без группы "популярность и дубликаты": 0.7096 (-0.0397)


**Выводы по абляции**

- Сильнее всего скор падает без таймингов (−0.06), без популярности объявлений и дубликатов (−0.04) и без окружения (−0.03).
- Без групп «положение в окне» и «содержание» скор даже чуть выше (+0.01). Разница сопоставима с разбросом между фолдами (±0.009), так что вывод о бесполезности этих групп пока ненадёжен: нужна проверка на большем числе дат разреза.
- При этом признаки из этих групп (возраст куки, начало и конец активности) стоят высоко в feature importance. Importance считает, как часто модель делает по признаку разбиения, а не насколько он улучшает качество, поэтому абляция здесь надёжнее.

## Финальная модель и сабмит

Обучаемся на всём train, усредняем предсказания по пяти сидам.

In [12]:
SEEDS = [0, 1, 2, 3, 4]

proba = np.zeros(len(X_test))
for seed in SEEDS:
    model = make_model(seed)
    model.fit(X_train[FEATURES], X_train.target)
    proba += model.predict_proba(X_test[FEATURES])[:, 1] / len(SEEDS)

submission = pd.DataFrame({'cookie_id': X_test.cookie_id, 'score': proba})
assert len(submission) == len(test)
assert submission.score.between(0, 1).all()
assert submission.cookie_id.is_unique

submission.to_csv('submission.csv', index=False)
print(submission.shape)
submission.head()

(4909, 2)


,cookie_id,score
0,ck_315fb710a0e371e7,0.000461
1,ck_a76ee3b3e3e522fd,0.201675
2,ck_94c9a4d382689e82,0.023249
3,ck_8eaf9509ad9462a0,0.000841
4,ck_9a88a5a989cb5bc6,0.019136


## Итог

- P@R70 = 0.75 ± 0.01 на трёх временных фолдах против 0.10 у baseline.
- Главные сигналы: ритм действий, окружение (user-agent, курсор) и популярность просматриваемых объявлений.
- Счётчик популярности по всем дням давал около 0.85 на валидации, но заглядывает в будущее относительно окна, поэтому не используется.

Что дальше: перепроверить группы с нулевым вкладом на большем числе фолдов, подобрать гиперпараметры на временной валидации, добавить признаки по последовательности действий (например, частые цепочки событий).